# Notebook 02 — Function Calling e Tool Use

**Settimana:** 2
**Blocco:** 3.2 — Function calling / tool use
**Tempo stimato:** 2.5-3.5 ore

## Cosa costruirai

Al termine di questo notebook avrai:
- Il loop completo di tool use implementato per OpenAI e Anthropic
- Un assistente con tre strumenti reali funzionanti
- Parallel tool use: il modello chiama due strumenti contemporaneamente
- Un registro di tutti gli strumenti invocati per sessione
- Un mini-framework riutilizzabile per registrare ed eseguire strumenti

## Prerequisiti

- Notebook 01 completato (client OpenAI e Anthropic inizializzati)
- `pip install openai anthropic python-dotenv requests wikipedia-api`

---
## Cella 0 — Setup

In [ ]:
import os
import json
import math
import datetime
import requests
from dotenv import load_dotenv
from openai import OpenAI
import anthropic

load_dotenv()

cliente_openai    = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
cliente_anthropic = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

print("Client inizializzati correttamente.")

---
## Cella 1 — Implementa i tre strumenti reali

Prima di occuparci dell'LLM, implementa le funzioni Python che saranno gli strumenti.
Sono funzioni normali — l'LLM non le tocca, le chiami solo tu.

I tre strumenti:
1. `obtener_fecha_hora` — restituisce la data e l'ora attuali
2. `calcular` — valuta espressioni matematiche in modo sicuro
3. `buscar_wikipedia` — restituisce il riassunto di un articolo di Wikipedia

In [ ]:
# ─── Strumento 1: data e ora attuali ───────────────────────────────────────

def obtener_fecha_hora(zona_horaria: str = "Europe/Rome") -> dict:
    """
    Restituisce la data e l'ora attuali.
    Nota: in produzione useresti pytz o zoneinfo per fusi orari reali.
    Qui semplifichiamo con UTC.
    """
    ahora = datetime.datetime.utcnow()
    return {
        "fecha":         ahora.strftime("%Y-%m-%d"),
        "hora":          ahora.strftime("%H:%M:%S"),
        "dia_semana":    ahora.strftime("%A"),
        "zona_horaria":  "UTC (approssimato)",
        "timestamp_iso": ahora.isoformat()
    }


# ─── Strumento 2: calcolatrice sicura ────────────────────────────────────────

def calcular(expresion: str) -> dict:
    """
    Valuta un'espressione matematica in modo sicuro.
    Permette: +, -, *, /, **, %, sqrt, sin, cos, tan, log, pi, e
    """
    # Whitelist delle funzioni consentite — NON usare MAI eval() senza restrizioni
    funciones_permitidas = {
        "sqrt":  math.sqrt,
        "sin":   math.sin,
        "cos":   math.cos,
        "tan":   math.tan,
        "log":   math.log,
        "log10": math.log10,
        "pi":    math.pi,
        "e":     math.e,
        "abs":   abs,
        "round": round,
    }
    try:
        resultado = eval(expresion, {"__builtins__": {}}, funciones_permitidas)
        return {
            "expresion": expresion,
            "resultado": resultado,
            "error":     None
        }
    except Exception as e:
        return {
            "expresion": expresion,
            "resultado": None,
            "error":     str(e)
        }


# ─── Strumento 3: ricerca su Wikipedia ────────────────────────────────────────

def buscar_wikipedia(tema: str, idioma: str = "it", oraciones: int = 3) -> dict:
    """
    Restituisce il riassunto di un articolo di Wikipedia.
    Usa l'API REST di Wikipedia (non richiede chiave).
    """
    url = f"https://{idioma}.wikipedia.org/api/rest_v1/page/summary/{requests.utils.quote(tema)}"
    try:
        respuesta = requests.get(url, timeout=5)
        respuesta.raise_for_status()
        datos = respuesta.json()

        # Taglia a N frasi
        extracto = datos.get("extract", "")
        oraciones_lista = extracto.split(". ")
        resumen = ". ".join(oraciones_lista[:oraciones]) + "."

        return {
            "tema":     tema,
            "titulo":   datos.get("title", tema),
            "resumen":  resumen,
            "url":      datos.get("content_urls", {}).get("desktop", {}).get("page", ""),
            "error":    None
        }
    except requests.exceptions.HTTPError:
        return {
            "tema":    tema,
            "resumen": None,
            "error":   f"Articolo '{tema}' non trovato su Wikipedia ({idioma})."
        }
    except Exception as e:
        return {
            "tema":    tema,
            "resumen": None,
            "error":   str(e)
        }


# ─── Test manuale dei tre strumenti ──────────────────────────────────
print("=== Strumento 1: data e ora ===")
print(json.dumps(obtener_fecha_hora(), indent=2, ensure_ascii=False))

print("\n=== Strumento 2: calcolatrice ===")
print(json.dumps(calcular("sqrt(144) + pi * 2"), indent=2))
print(json.dumps(calcular("1 / 0"), indent=2))  # caso di errore

print("\n=== Strumento 3: Wikipedia ===")
print(json.dumps(buscar_wikipedia("Intelligenza artificiale"), indent=2, ensure_ascii=False))

---
## Cella 2 — Definisci gli strumenti in formato JSON Schema

Ora descrivi gli stessi strumenti nel formato che l'LLM capisce.
Ricorda: la `description` è ciò che il modello legge per decidere quando usarli.

In [ ]:
# ─── Definizioni per OpenAI ─────────────────────────────────────────────────

HERRAMIENTAS_OPENAI = [
    {
        "type": "function",
        "function": {
            "name": "obtener_fecha_hora",
            "description": (
                "Restituisce la data e l'ora attuali. "
                "Usala quando l'utente chiede che giorno è oggi, che ore sono, "
                "il giorno della settimana, o qualsiasi riferimento al momento presente. "
                "NON usarla per date storiche o future."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "zona_horaria": {
                        "type": "string",
                        "description": "Fuso orario in formato IANA, es. 'Europe/Rome'. Default: Europe/Rome."
                    }
                },
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calcular",
            "description": (
                "Valuta espressioni matematiche. "
                "Usala per qualsiasi calcolo numerico: aritmetica, radici, trigonometria, logaritmi. "
                "L'espressione deve essere in sintassi Python matematica, es. 'sqrt(25) + 3*4'. "
                "NON usarla per domande concettuali sulla matematica."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "expresion": {
                        "type": "string",
                        "description": "Espressione matematica valida in Python, es. '2**10' o 'sqrt(144)'"
                    }
                },
                "required": ["expresion"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "buscar_wikipedia",
            "description": (
                "Cerca e restituisce il riassunto di un articolo di Wikipedia. "
                "Usala quando ti serve un'informazione enciclopedica su persone, luoghi, "
                "concetti, eventi storici o scientifici. "
                "NON usarla per eventi molto recenti (Wikipedia potrebbe non essere aggiornata) "
                "né per opinioni o raccomandazioni."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "tema": {
                        "type": "string",
                        "description": "Termine di ricerca così come apparirebbe nel titolo dell'articolo di Wikipedia"
                    },
                    "idioma": {
                        "type": "string",
                        "enum": ["es", "en", "fr", "de", "it", "pt"],
                        "description": "Codice lingua di Wikipedia. Default: 'it' (italiano)"
                    },
                    "oraciones": {
                        "type": "integer",
                        "description": "Numero di frasi del riassunto. Tra 1 e 5. Default: 3."
                    }
                },
                "required": ["tema"]
            }
        }
    }
]


# ─── Definizioni per Anthropic ─────────────────────────────────────────────
# La logica è identica; cambia solo la struttura del JSON.

HERRAMIENTAS_ANTHROPIC = [
    {
        "name": "obtener_fecha_hora",
        "description": (
            "Restituisce la data e l'ora attuali. "
            "Usala quando l'utente chiede che giorno è oggi, che ore sono, "
            "il giorno della settimana, o qualsiasi riferimento al momento presente."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "zona_horaria": {
                    "type": "string",
                    "description": "Fuso orario in formato IANA. Default: Europe/Rome."
                }
            },
            "required": []
        }
    },
    {
        "name": "calcular",
        "description": (
            "Valuta espressioni matematiche. Usala per qualsiasi calcolo numerico. "
            "L'espressione deve essere in sintassi Python matematica."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "expresion": {
                    "type": "string",
                    "description": "Espressione matematica valida in Python"
                }
            },
            "required": ["expresion"]
        }
    },
    {
        "name": "buscar_wikipedia",
        "description": (
            "Cerca e restituisce il riassunto di un articolo di Wikipedia. "
            "Usala per informazioni enciclopediche su persone, luoghi o concetti."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "tema": {
                    "type": "string",
                    "description": "Termine di ricerca"
                },
                "idioma": {
                    "type": "string",
                    "enum": ["es", "en", "fr", "de", "it", "pt"],
                    "description": "Codice lingua. Default: 'it'"
                },
                "oraciones": {
                    "type": "integer",
                    "description": "Numero di frasi del riassunto (1-5). Default: 3."
                }
            },
            "required": ["tema"]
        }
    }
]


# ─── Dispatcher: mappa nome dello strumento → funzione Python ─────────────────

FUNCIONES_DISPONIBLES = {
    "obtener_fecha_hora": obtener_fecha_hora,
    "calcular":           calcular,
    "buscar_wikipedia":   buscar_wikipedia,
}

def ejecutar_herramienta(nombre: str, argumentos: dict) -> str:
    """Esegue lo strumento per nome e restituisce il risultato come stringa JSON."""
    if nombre not in FUNCIONES_DISPONIBLES:
        return json.dumps({"error": f"Strumento '{nombre}' non trovato."})

    funcion  = FUNCIONES_DISPONIBLES[nombre]
    resultado = funcion(**argumentos)
    return json.dumps(resultado, ensure_ascii=False)


print(f"Strumenti registrati: {list(FUNCIONES_DISPONIBLES.keys())}")

---
## Cella 3 — Loop completo di tool use con OpenAI

Implementa il loop che:
1. Chiama il modello con gli strumenti disponibili
2. Se il modello richiede strumenti, li esegue
3. Restituisce i risultati al modello
4. Ripete finché il modello non genera una risposta di testo finale

Studia il flusso prima di eseguire. Quante chiamate HTTP vengono fatte per una domanda che usa uno strumento?

In [ ]:
def asistente_con_herramientas_openai(
    pregunta: str,
    system_prompt: str = "Sei un assistente utile con accesso a strumenti. Rispondi sempre in italiano.",
    verbose: bool = True
) -> str:
    """
    Esegue il loop completo di tool use con OpenAI.
    Restituisce la risposta finale in testo.
    """
    historial = [
        {"role": "system",  "content": system_prompt},
        {"role": "user",    "content": pregunta}
    ]

    iteracion = 0
    max_iteraciones = 5  # per evitare loop infiniti

    while iteracion < max_iteraciones:
        iteracion += 1
        if verbose:
            print(f"\n[Iterazione {iteracion}] Chiamando il modello...")

        respuesta = cliente_openai.chat.completions.create(
            model="gpt-4o-mini",
            messages=historial,
            tools=HERRAMIENTAS_OPENAI,
            tool_choice="auto"
        )

        mensaje = respuesta.choices[0].message

        # ── Caso A: il modello vuole usare strumenti ──────────────────────
        if mensaje.tool_calls:
            # Aggiunge la risposta del modello alla cronologia
            historial.append(mensaje)

            # Esegue ogni strumento e aggiunge il risultato alla cronologia
            for tool_call in mensaje.tool_calls:
                nombre    = tool_call.function.name
                argumentos = json.loads(tool_call.function.arguments)

                if verbose:
                    print(f"  → Eseguo: {nombre}({argumentos})")

                resultado = ejecutar_herramienta(nombre, argumentos)

                if verbose:
                    print(f"  ← Risultato: {resultado}")

                historial.append({
                    "role":         "tool",
                    "tool_call_id": tool_call.id,
                    "content":      resultado
                })

            # Continua il loop: il modello ora ha i risultati
            continue

        # ── Caso B: il modello restituisce una risposta di testo → fine del loop ─
        texto_final = mensaje.content
        if verbose:
            print(f"\n[Risposta finale dopo {iteracion} iterazione/i]")
        return texto_final

    return "[Errore: raggiunto il numero massimo di iterazioni senza risposta finale]"


# ─── Test ─────────────────────────────────────────────────────────────────

# Test 1: domanda che richiede lo strumento data/ora
print("\n" + "="*60)
print("DOMANDA 1: data e ora")
print("="*60)
respuesta1 = asistente_con_herramientas_openai("Che giorno della settimana è oggi?")
print(f"\nRisposta: {respuesta1}")

# Test 2: domanda che richiede la calcolatrice
print("\n" + "="*60)
print("DOMANDA 2: calcolo")
print("="*60)
respuesta2 = asistente_con_herramientas_openai("Quanto vale la radice quadrata di 17161?")
print(f"\nRisposta: {respuesta2}")

# Test 3: domanda che NON richiede strumenti
print("\n" + "="*60)
print("DOMANDA 3: senza strumenti")
print("="*60)
respuesta3 = asistente_con_herramientas_openai("Cos'è un transformer nell'IA?")
print(f"\nRisposta: {respuesta3}")

---
## Cella 4 — Loop completo con Anthropic

Lo stesso loop, adattato alle differenze dell'API di Anthropic.
Confronta questo codice con quello della cella precedente: le differenze sono esattamente
quelle che hai documentato nella dispensa.

In [ ]:
def asistente_con_herramientas_anthropic(
    pregunta: str,
    system_prompt: str = "Sei un assistente utile con accesso a strumenti. Rispondi sempre in italiano.",
    verbose: bool = True
) -> str:
    """
    Esegue il loop completo di tool use con Anthropic.
    """
    historial   = [{"role": "user", "content": pregunta}]
    iteracion   = 0
    max_iteraciones = 5

    while iteracion < max_iteraciones:
        iteracion += 1
        if verbose:
            print(f"\n[Iterazione {iteracion}] Chiamando il modello...")

        respuesta = cliente_anthropic.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1024,
            system=system_prompt,
            tools=HERRAMIENTAS_ANTHROPIC,
            messages=historial
        )

        # ── Caso A: il modello vuole usare strumenti ──────────────────────
        # In Anthropic: stop_reason == "tool_use" e ci sono blocchi tool_use in content
        if respuesta.stop_reason == "tool_use":

            # Aggiunge la risposta del modello alla cronologia
            historial.append({
                "role":    "assistant",
                "content": respuesta.content  # lista di blocchi (text + tool_use)
            })

            # Raccoglie i risultati di tutti gli strumenti
            resultados_herramientas = []

            for bloque in respuesta.content:
                if bloque.type == "tool_use":
                    nombre     = bloque.name
                    argumentos = bloque.input

                    if verbose:
                        print(f"  → Eseguo: {nombre}({argumentos})")

                    resultado = ejecutar_herramienta(nombre, argumentos)

                    if verbose:
                        print(f"  ← Risultato: {resultado}")

                    resultados_herramientas.append({
                        "type":        "tool_result",
                        "tool_use_id": bloque.id,
                        "content":     resultado
                    })

            # I risultati vanno in un unico messaggio di ruolo "user"
            historial.append({
                "role":    "user",
                "content": resultados_herramientas
            })
            continue

        # ── Caso B: risposta di testo finale ────────────────────────────────
        texto_final = "".join(
            bloque.text for bloque in respuesta.content
            if hasattr(bloque, "text")
        )
        if verbose:
            print(f"\n[Risposta finale dopo {iteracion} iterazione/i]")
        return texto_final

    return "[Errore: raggiunto il numero massimo di iterazioni senza risposta finale]"


# ─── Gli stessi test di OpenAI — confronta i risultati ──────────────────

print("\n" + "="*60)
print("DOMANDA 1: data e ora (Anthropic)")
print("="*60)
respuesta1 = asistente_con_herramientas_anthropic("Che giorno della settimana è oggi?")
print(f"\nRisposta: {respuesta1}")

print("\n" + "="*60)
print("DOMANDA 2: Wikipedia (Anthropic)")
print("="*60)
respuesta2 = asistente_con_herramientas_anthropic("Chi era Alan Turing e cosa ha contribuito all'informatica?")
print(f"\nRisposta: {respuesta2}")

---
## Cella 5 — Parallel tool use

Lancia una domanda che richiede due strumenti indipendenti contemporaneamente.
Osserva che il modello genera DUE tool_calls in un'unica risposta,
e tu le esegui entrambe prima di richiamare il modello.

In [ ]:
PREGUNTA_PARALELA = (
    "Ho bisogno di due cose: prima, dimmi quanti giorni mancano al 31 dicembre "
    "(usa lo strumento data per sapere la data di oggi e calcola con la calcolatrice). "
    "Seconda, dimmi chi era Marie Curie in due frasi."
)

print("Domanda:", PREGUNTA_PARALELA)
print("\nEseguo con OpenAI (verbose=True per vedere il parallel tool use):")
print("="*60)

respuesta_paralela = asistente_con_herramientas_openai(PREGUNTA_PARALELA, verbose=True)
print(f"\nRisposta finale:\n{respuesta_paralela}")

**Domanda di riflessione:** quante iterazioni del loop ti aspettavi? Quante ne sono avvenute realmente? Perché?

La tua risposta: ...

---
## Sfida finale — Mini-framework con registro dell'uso degli strumenti

Costruisci una versione migliorata dell'assistente che:
1. Registra un log di ogni strumento invocato (nome, argomenti, risultato, timestamp)
2. Alla fine della sessione, stampa un riepilogo: quante volte è stato usato ogni strumento
3. Funziona con entrambi i provider (openai / anthropic) tramite un parametro

Leggi lo scheletro, completa i `# TODO` e provalo con almeno 5 domande diverse.

In [ ]:
from collections import defaultdict
from typing import Literal

class AsistenteConHerramientas:
    """
    Assistente multi-provider con registro dell'uso degli strumenti.
    """

    def __init__(
        self,
        proveedor: Literal["openai", "anthropic"] = "openai",
        system_prompt: str = "Sei un assistente utile. Rispondi sempre in italiano."
    ):
        self.proveedor    = proveedor
        self.system_prompt = system_prompt
        self.log_herramientas: list[dict] = []       # registro completo
        self.contador     = defaultdict(int)          # nome → numero di volte usato

    def _ejecutar_y_registrar(self, nombre: str, argumentos: dict) -> str:
        """Esegue lo strumento e registra l'invocazione."""
        resultado = ejecutar_herramienta(nombre, argumentos)

        self.log_herramientas.append({
            "timestamp":  datetime.datetime.utcnow().isoformat(),
            "herramienta": nombre,
            "argumentos": argumentos,
            "resultado":  resultado
        })
        self.contador[nombre] += 1

        return resultado

    def preguntar(self, pregunta: str, verbose: bool = False) -> str:
        """Invia una domanda e restituisce la risposta finale."""
        # TODO: implementa qui il loop completo.
        # Usa self.proveedor per decidere quale client e quali strumenti usare.
        # Usa self._ejecutar_y_registrar() invece di ejecutar_herramienta() direttamente.
        #
        # Suggerimento: puoi riutilizzare quasi tutto il codice delle celle 3 e 4,
        # sostituendo ejecutar_herramienta() con self._ejecutar_y_registrar()

        raise NotImplementedError("Implementa questo metodo")

    def resumen_uso(self) -> None:
        """Stampa il riepilogo dell'uso degli strumenti della sessione."""
        print(f"\n{'='*40}")
        print("RIEPILOGO USO STRUMENTI")
        print(f"{'='*40}")
        print(f"Totale invocazioni: {sum(self.contador.values())}")
        for nombre, veces in sorted(self.contador.items(), key=lambda x: -x[1]):
            print(f"  {nombre}: {veces} volta/e")
        print(f"{'='*40}")

    def exportar_log(self, ruta: str = "herramientas_log.json") -> None:
        """Salva il log completo in un file JSON."""
        with open(ruta, "w", encoding="utf-8") as f:
            json.dump(self.log_herramientas, f, indent=2, ensure_ascii=False)
        print(f"Log salvato in {ruta} ({len(self.log_herramientas)} voci)")


# ─── Test del mini-framework ────────────────────────────────────────────────
# Decommenta dopo aver implementato il metodo preguntar()

# asistente = AsistenteConHerramientas(proveedor="openai")

# preguntas = [
#     "Che ore sono adesso?",
#     "Quanto fa 2 elevato a 32?",
#     "Dimmi qualcosa sulla Rivoluzione francese.",
#     "Che giorno è oggi e quanto vale sqrt(256)?",
#     "Cos'è Python secondo Wikipedia?"
# ]

# for pregunta in preguntas:
#     print(f"\nDomanda: {pregunta}")
#     respuesta = asistente.preguntar(pregunta)
#     print(f"Risposta: {respuesta}")

# asistente.resumen_uso()
# asistente.exportar_log()

### Estensioni opzionali

1. **Timeout per strumento:** se `buscar_wikipedia` impiega più di 3 secondi, restituisci un errore invece di bloccare.

2. **Nuovo strumento:** aggiungi `convertir_divisas` che chiami l'API gratuita di `exchangerate-api.com` e registra il suo uso nel sistema.

3. **Cronologia persistente:** salva la cronologia completa della conversazione in un JSON al termine, non solo il log degli strumenti.

In [ ]:
# Spazio per le estensioni opzionali

---
## Autovalutazione della settimana

1. **Cosa ha funzionato bene?**
   ...

2. **Cosa non hai capito del tutto?**
   ...

3. **Cosa cambieresti la prossima settimana?**
   ...

4. **Tempo reale dedicato:**
   ...

---
*Notebook 02 — Settimana 2 del Piano di Formazione AI Engineering*